In [5]:
import pandas as pd
import yaml
import subprocess

In [6]:
REGENERATE_RULES = False 
RULES_YML_PATH = "data/asca_rules.yml"
RULES_CSV_PATH = "data/asca_rules.csv"
DF_COLS = ["section_index", "section_name", "rule_id", "step_index", "c_index", "input", "output", "env", "exception"]

def split_condensed_rule(rule: dict) -> list[dict]:
    if "input" not in rule or "output" not in rule:
        raise ValueError("Input or output not found in rule")
    inputs = rule["input"].strip().split(" ")
    outputs = rule["output"].strip().split(" ")
    if (len(inputs) == len(outputs)) and len(inputs) > 1:
        return [{**rule, "input": i, "output": o} for i, o in zip(inputs, outputs)]
    return [rule]

if REGENERATE_RULES:
    cmd = ["uv", "run", "compile_rules", "--compiler-config", "../config/pipeline/compiler_config.yml", "--compiled-out", RULES_YML_PATH]

    output = subprocess.run(
        cmd, capture_output=True, timeout=60, text=True, check=True,
    )
    output.check_returncode()

    with open(RULES_YML_PATH, "r") as f:
        rules_doc = yaml.load(f, Loader=yaml.SafeLoader)

    rules = []
    for s in rules_doc["sections"]:
        for r in s["rules"]:
            if r.get("status", "") == "skipped" or "compile_error" in r:
                continue
            if "steps" not in r:
                for j, rule in enumerate(split_condensed_rule(r)):
                    rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": 0, "c_index": j, **rule})
            else:
                for i,step in enumerate(r["steps"]):
                    for j, rule in enumerate(split_condensed_rule(step)):
                        rules.append({"section_index": s["index"], "section_name": s["section"], "step_index": i, "c_index": j, "rule_id": r["rule_id"], **rule})

    rules_df = pd.DataFrame(rules)[DF_COLS]
    rules_df.to_csv(RULES_CSV_PATH, index=False)
else:
    rules_df = pd.read_csv(RULES_CSV_PATH)

In [7]:
print(rules_df.shape)
rules_df.head(10)

(11994, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,F,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,F,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
8,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-aː,0,0,"{a,a:[+long]}","{e,e:[+long]}","_{ʕ,q}$",NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN


In [8]:
segments = pd.concat([rules_df["input"], rules_df["output"]])
contexts = pd.concat([rules_df["env"], rules_df["exception"]]).dropna()

min_segment_counts = 5
min_context_counts = 5

segments_counts = segments.value_counts()
contexts_counts = contexts.value_counts()

sample_segments = set(segments_counts[(segments_counts >= min_segment_counts)].index)
sample_contexts = set(contexts_counts[(contexts_counts >= min_context_counts)].index)

usable_inputs = rules_df["input"].isin(sample_segments)
usable_outputs = rules_df["output"].isin(sample_segments)
usable_envs = rules_df["env"].isin(sample_contexts) | rules_df["env"].isna()
usable_exceptions = rules_df["exception"].isin(sample_contexts) | rules_df["exception"].isna()

usable_rules_df = rules_df[usable_inputs & usable_outputs & usable_envs & usable_exceptions]
print(usable_rules_df.shape)
usable_rules_df.head(20)

(6724, 9)


,section_index,section_name,rule_id,step_index,c_index,input,output,env,exception
0,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,0,dz,ʒ,NaN,NaN
1,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,1,ʃ,F,NaN,NaN
2,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dz-ʃ-tʃ,0,2,tʃ,F,NaN,NaN
3,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,0,0,dʒ,tʃ,NaN,NaN
4,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-dʒ,1,0,tʃ,ʃ,NaN,NaN
5,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-t,0,0,t,∅,_s#,NaN
6,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-ɬ,0,0,ɬ,l,NaN,NaN
7,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-f,0,0,f,p,NaN,NaN
9,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,0,q,ʔ,NaN,NaN
10,6.1,Proto-Afro-Asiatic to Proto-Omotic,Proto-Omotic-q-ʕ,0,1,ʕ,h,NaN,NaN
